In [1]:
from pathlib import Path

import numpy as np
from tqdm import tqdm
import matplotlib.pyplot as plt
# torch (original name before being moved to Python and becoming PyTorch) used to 
# create tensors and store all numerical values, including raw data, weights and biases
import torch
from torch.utils.data import DataLoader, TensorDataset, random_split # importing also random_split for validation
# torch.nn is to make the weight and bias tensors part of the neural network
import torch.nn as nn
# torch.nn.functional is used to import the activation functions
import torch.nn.functional as F


In [2]:
# for tracking we try both :)
from torch.utils.tensorboard import SummaryWriter
import wandb

In [3]:

"""
README FIRST

The below code is a template for the solution. You can change the code according
to your preferences, but the testing function has to save the output of your 
model on the test data as it does in this template. This output must be submitted.

Replace the dummy code with your own code in the TODO sections.

We also encourage you to use tensorboard or wandb to log the training process
and the performance of your model. This will help you to debug your model and
to understand how it is performing. But the template does not include this
functionality.
Link for wandb:
https://docs.wandb.ai/quickstart/
Link for tensorboard: 
https://pytorch.org/tutorials/recipes/recipes/tensorboard_with_pytorch.html
"""

# The device is automatically set to GPU if available, otherwise CPU
# If you want to force the device to CPU, you can change the line to
# device = torch.device("cpu")

# If you have a Mac consult the following link:
# https://pytorch.org/docs/stable/notes/mps.html

# It is important that your model and all data are on the same device.
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
torch.cuda.is_available()


True

In [4]:

def load_data(**kwargs):
    """
    Get the training and test data. The data files are assumed to be in the
    same directory as this script.

    Args:
    - kwargs: Additional arguments that you might find useful - not necessary

    Returns:
    - train_data_input: Tensor[N_train_samples, C, H, W]
    - train_data_label: Tensor[N_train_samples, C, H, W]
    - test_data_input: Tensor[N_test_samples, C, H, W]
    where N_train_samples is the number of training samples, N_test_samples is
    the number of test samples, C is the number of channels (1 for grayscale),
    H is the height of the image, and W is the width of the image.
    """
    # Load the training data
    train_data = np.load("../train_data.npz")["data"]

    # Make the training data a tensor
    train_data = torch.tensor(train_data, dtype=torch.float32) / 255.0

    # Load the test data
    test_data_input = np.load("../test_data.npz")["data"]

    # Make the test data a tensor
    test_data_input = torch.tensor(test_data_input, dtype=torch.float32) / 255.0

    ########################################
    # TODO: Given the original training images, create the input images and the
    # label images to train your model. 
    # Replace the two placholder lines below (which currently just copy the
    # training data) with your own implementation.

    # as per the test images seen below, a square mask of 64 pixels from x = 10, y = 10 to x = 18, y = 18 will be applied
    train_data_label = train_data.clone()
    train_data_input = train_data.clone()
    train_data_input[:, :, 10:18, 10:18] = 0.0

    # Visualize the training data if needed
    # Set to False if you don't want to save the images
    if True:
        # Create the output directory if it doesn't exist
        if not Path("train_image_output").exists():
            Path("train_image_output").mkdir()
        for i in tqdm(range(20), desc="Plotting train images"):
            # Show the training and the target image side by side
            plt.subplot(1, 2, 1)
            plt.imshow(train_data_input[i].squeeze(), cmap="gray")
            plt.title("Training Input")
            plt.subplot(1, 2, 2)
            plt.title("Training Label")
            plt.imshow(train_data_label[i].squeeze(), cmap="gray")

            plt.savefig(f"train_image_output/image_{i}.png")
            plt.close()

# having a look at the Test images to understand if they have been masked and denoised, conclusion: masked from 10 to 18 pixels x and y axis
        if True:
        # Create the output directory if it doesn't exist
            if not Path("test_image_output").exists():
                Path("test_image_output").mkdir()
            for i in tqdm(range(20), desc="Plotting test images"):
                # Show the training and the target image side by side
                plt.subplot(1, 2, 1)
                plt.imshow(test_data_input[i].squeeze(), cmap="gray")
                plt.title("Test Input")

                plt.savefig(f"test_image_output/image_{i}.png")
                plt.close()

    return train_data_input, train_data_label, test_data_input



In [5]:

def training(train_data_input, train_data_label, **kwargs):
    """
    Train the model. Fill in the details of the data loader, the loss function,
    the optimizer, and the training loop.

    Args:
    - train_data_input: Tensor[N_train_samples, C, H, W]
    - train_data_label: Tensor[N_train_samples, C, H, W]
    - kwargs: Additional arguments that you might find useful - not necessary

    Returns:
    - model: torch.nn.Module
    """
    model = Model()
    model.train()
    model.to(device)

    # TODO: Dummy criterion - change this to the correct loss function
    # https://pytorch.org/docs/stable/nn.html#loss-functions
    criterion = nn.MSELoss() # would be nn.CrossEntropyLoss for integer labels
    # L1 loss may be better than MSE because outliers are less important, so the image results to be less blurry, 
    # however MSE is the final to predict, so MSE is used, a combintation of the two may also be good
    # TODO: Dummy optimizer - change this to a more suitable optimizer
    optimizer = torch.optim.Adam(model.parameters(), lr = 0.001,) 
    # the scheduler below drop the LR by 50% if the Val Loss doesn't improve for 5 epochs
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=5)
    # optimiser could be optim.SGD(....,momentum = 0.9) as per PyTorch tutorial and Medium article for integer values maybe, but this resulted to be better

    # TODO: Correctly setup the dataloader - the below is just a placeholder
    # Also consider that you might not want to use the entire dataset for
    # training alone
    # (batch_size needs to be changed)
    batch_size = 64 # PyTorch tutorial has 4, tried 4 first, but each batch was probably too noise, then increased
    dataset = TensorDataset(train_data_input, train_data_label)
    # Consider the shuffle parameter and other parameters of the DataLoader
    # class (see
    # https://pytorch.org/docs/stable/data.html#torch.utils.data.DataLoader)
    train_size = int(0.95 * len(dataset)) # 95%
    val_size = len(dataset) - train_size # 5%
    # tried a 80/20 split beforehand but decided to use 95/5 instead to use more images for training and 
    # the validation set remains anyway large enough.
    
    train_dataset, val_dataset = random_split(dataset, [train_size, val_size])

    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)

    # Training loop
    # TODO: Modify the training loop in case you need to
    # splitting data into training and validation, just to try to avoid overfitting
    # TODO: The value of n_epochs is just a placeholder and likely needs to be
    # changed

    # initialising tensorboard
    writer = SummaryWriter(log_dir="runs/inpainting_MSELoss_bencoder")
    
    n_epochs = 100


    # initialising wandb
    wandb.init(
        project="digit-image-inpainting", 
        name="MSE-Loss_BottleneckEncoder",
        config={
            "learning_rate": 0.001,
            "architecture": "U-NET",
            "epochs": n_epochs,
            "batch_size": batch_size,
            "loss_function": "MSELoss"
        }
    )

    best_val_loss = float('inf')
    patience = 10 # how many epochs to wait for an improvement
    patience_counter = 0   
    
    for epoch in range(n_epochs):
        model.train() # just to be sure
        running_train_loss = 0.0

        for x, y in tqdm(
            train_loader, desc=f"Training Epoch {epoch}", leave=False
        ):
            x, y = x.to(device), y.to(device)
            optimizer.zero_grad()
            output = model(x)
            pred_patch = output[:,:, 10:18, 10:18] # since it is only the central square missing, that is what we compute the loss on
            true_patch = y[:, :, 10:18, 10:18]
            loss = criterion(pred_patch, true_patch) # focusing only on the masked patch
            loss.backward()
            optimizer.step()

            running_train_loss += loss.item() * x.size(0)

        avg_train_loss = running_train_loss / len(train_dataset)

        model.eval() # putting model in eval mode (freezes dropout/batchnorm)
        running_val_loss = 0.0
        
        with torch.no_grad(): # not tracking gradients for validation
            for val_x, val_y in val_loader:
                val_x, val_y = val_x.to(device), val_y.to(device)
                val_output = model(val_x)
                val_pred_patch = val_output[:, :, 10:18, 10:18]
                val_true_patch = val_y[:, :, 10:18, 10:18]
                
                val_loss = criterion(val_pred_patch, val_true_patch)
                running_val_loss += val_loss.item() * val_x.size(0)
                
        avg_val_loss = running_val_loss / len(val_dataset)

        # making the scheduler (the lr of the optimiser decrease) if the patience has been reached.
        scheduler.step(avg_val_loss)

        print(f"Epoch {epoch} | Train Loss: {avg_train_loss:.6f} | Val Loss: {avg_val_loss:.6f}")


        # logging numbers to TensorBoard
        writer.add_scalar('Loss/Train', avg_train_loss, epoch)
        writer.add_scalar('Loss/Validation', avg_val_loss, epoch)
        
        # logging images to TensorBoard (Using the last batch from validation)
        writer.add_images('Validation/Masked_Input', val_x, epoch)
        writer.add_images('Validation/Generated_Output', val_output, epoch)
        writer.add_images('Validation/Ground_Truth', val_y, epoch)

        # 3. Logging to wanb
        wandb.log({
            "Train Loss": avg_train_loss,
            "Val Loss": avg_val_loss,
            "Epoch": epoch,
            "Generated Examples": [wandb.Image(img) for img in val_output] # Logs the batch as images
        })

# implementing a patience_counter that makes the loop stop if in 10 epochs the val loss does not decrease


        if avg_val_loss < best_val_loss:
            best_val_loss = avg_val_loss
            patience_counter = 0 # Reset counter because we got a new best         
            # save the best model weights so we keep the best version
            torch.save(model.state_dict(), "best_bencoder_model.pth")
        else:
            patience_counter += 1 # adding one to the patience counter because of no improvement
            print(f"No improvement in Val Loss. Patience: {patience_counter}/{patience}")

        if patience_counter >= patience:
            print(f"Early stopping triggered at Epoch {epoch}!")
            # Load the best weights back into the model before returning it
            model.load_state_dict(torch.load("best_bencoder_model.pth"))
            break

    # closing the tensorboard writer and wandb
    writer.close()
    wandb.finish()

    return model



In [6]:

# TODO: define a model. Here, a basic MLP model is defined. You can completely
# change this model - and are encouraged to do so.
   


class Model(nn.Module):
    def __init__(self):
        super().__init__()
        
        # 1. ENCODER (Lighter: 16 and 32 channels)
        self.encoder = nn.Sequential(
            nn.Conv2d(1, 16, kernel_size=3, stride=2, padding=1), # 28x28 -> 14x14
            nn.BatchNorm2d(16),
            nn.LeakyReLU(0.2),
            nn.Conv2d(16, 32, kernel_size=3, stride=2, padding=1), # 14x14 -> 7x7
            nn.BatchNorm2d(32),
            nn.LeakyReLU(0.2)
        )
        
        # 2. BOTTLENECK (Lighter: 1568 -> 128 -> 1568)
        # 32 channels * 7 * 7 = 1568 features.
        self.bottleneck = nn.Sequential(
            nn.Flatten(),
            nn.Linear(1568, 128),
            nn.BatchNorm1d(128),
            nn.LeakyReLU(0.2),
            nn.Linear(128, 1568),
            nn.BatchNorm1d(1568),
            nn.LeakyReLU(0.2)
        )
        
        # 3. DECODER (Lighter: 32 and 16 channels)
        self.decoder = nn.Sequential(
            nn.ConvTranspose2d(32, 16, kernel_size=3, stride=2, padding=1, output_padding=1), # 7x7 -> 14x14
            nn.BatchNorm2d(16),
            nn.LeakyReLU(0.2),
            nn.ConvTranspose2d(16, 1, kernel_size=3, stride=2, padding=1, output_padding=1), # 14x14 -> 28x28
            nn.Sigmoid()
        )

    def forward(self, x):
        x = self.encoder(x)
        x = self.bottleneck(x)
        x = x.view(x.size(0), 32, 7, 7) # Reshape back to the new 32-channel 2D map
        x = self.decoder(x)
        return x


In [7]:

def testing(model, test_data_input):
    """
    Uses your model to predict the ouputs for the test data. Saves the outputs
    as a binary file. This file needs to be submitted. This function does not
    need to be modified except for setting the batch_size value. If you choose
    to modify it otherwise, please ensure that the generating and saving of the
    output data is not modified.

    Args:
    - model: torch.nn.Module
    - test_data_input: Tensor
    """
    model.eval()
    model.to(device)

    with torch.no_grad():
        test_data_input = test_data_input.to(device)
        # Predict the output batch-wise to avoid memory issues
        test_data_output = []
        # TODO: You can increase or decrease this batch size depending on your
        # memory requirements of your computer / model
        # This will not affect the performance of the model and your score
        batch_size = 64
        for i in tqdm(
            range(0, test_data_input.shape[0], batch_size),
            desc="Predicting test output",
        ):
            output = model(test_data_input[i : i + batch_size])
            test_data_output.append(output.cpu())
        test_data_output = torch.cat(test_data_output)

    # Ensure the output has the correct shape
    assert test_data_output.shape == test_data_input.shape, (
        f"Expected shape {test_data_input.shape}, but got "
        f"{test_data_output.shape}."
        "Please ensure the output has the correct shape."
        "Without the correct shape, the submission cannot be evaluated and "
        "will hence not be valid."
    )

    # Save the output
    test_data_output = test_data_output.numpy()
    # Ensure all values are in the range [0, 255]
    test_data_output = test_data_output * 255.0 # added because of comment above and the Sigmoid function is used at the end of model
    save_data_clipped = np.clip(test_data_output, 0, 255)
    # Convert to uint8
    # Ensure your model outputs values in the [0, 255] range before this step! If you normalized your data to [0, 1], you must multiply by 255 before saving.
    save_data_uint8 = save_data_clipped.astype(np.uint8)
    # Loss is only computed on the masked area - so set the rest to 0 to save
    # space
    save_data = np.zeros_like(save_data_uint8)
    save_data[:, :, 10:18, 10:18] = save_data_uint8[:, :, 10:18, 10:18]

    np.savez_compressed(
        "submit_this_test_data_output_MSE_bencoder.npz", data=save_data)

    # You can plot the output if you want
    # Set to False if you don't want to save the images
    if True:
        # Create the output directory if it doesn't exist
        if not Path("test_image_output_MSE_bencoder").exists():
            Path("test_image_output_MSE_bencoder").mkdir()
        for i in tqdm(range(20), desc="Plotting test images"):
            # Show the training and the target image side by side
            input_np = test_data_input.cpu().numpy() * 255.0
            # just stitching the masked predicted part, back in the input (otherwise 
            # the output would have the context also predicted, which is not assessed given in the input)
            stitched_image = input_np[i].copy()
            stitched_image[:, 10:18, 10:18] = test_data_output[i, :, 10:18, 10:18]

            plt.subplot(1, 2, 1)
            plt.title("Test Input")
            plt.imshow(input_np[i].squeeze(), cmap="gray")
            plt.subplot(1, 2, 2)
            plt.imshow(stitched_image.squeeze(), cmap="gray")
            plt.title("Test Output")

            plt.savefig(f"test_image_output_MSE_bencoder/image_{i}.png")
            plt.close()


In [8]:


def main():
    seed = 0
    # Reproducibility
    torch.manual_seed(seed)
    np.random.seed(seed)
    torch.backends.cudnn.deterministic = True

    # You don't need to change the code below
    # Load the data
    train_data_input, train_data_label, test_data_input = load_data()
    # Train the model
    model = training(train_data_input, train_data_label)

    # Test the model (this also generates the submission file)
    # The name of the submission file is submit_this_test_data_output.npz
    testing(model, test_data_input)

    return None


if __name__ == "__main__":
    main()


Plotting test images: 100%|██████████| 20/20 [00:01<00:00, 12.48it/s]
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from C:\Users\carlo\_netrc.
wandb: Currently logged in as: carlo-rossetto (crossetto-eth-z-rich) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Epoch 0 | Train Loss: 0.086476 | Val Loss: 0.070950


Epoch 1 | Train Loss: 0.070035 | Val Loss: 0.066735


Epoch 2 | Train Loss: 0.065739 | Val Loss: 0.063784


Epoch 3 | Train Loss: 0.062983 | Val Loss: 0.062668


...

Epoch 26 | Train Loss: 0.044584 | Val Loss: 0.059879
No improvement in Val Loss. Patience: 8/10


Epoch 27 | Train Loss: 0.044246 | Val Loss: 0.059619
No improvement in Val Loss. Patience: 9/10


Epoch 28 | Train Loss: 0.044072 | Val Loss: 0.059967
No improvement in Val Loss. Patience: 10/10
Early stopping triggered at Epoch 28!


Epoch,▁▁▁▂▂▂▃▃▃▃▃▄▄▄▅▅▅▅▅▆▆▆▇▇▇▇▇██
Train Loss,█▅▅▄▄▄▃▃▃▃▃▃▂▂▂▂▂▂▂▂▂▂▂▂▂▁▁▁▁
Val Loss,█▅▄▃▃▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▂▁▁▂▂▁▁▁▁
Epoch,28
Train Loss,0.04407
Val Loss,0.05997


Plotting test images: 100%|██████████| 20/20 [00:07<00:00,  2.86it/s]
